# Vextor – training su Kaggle

Addestra il tuo modello per siti web partendo da **Qwen2.5-Coder** e lo esporta in **GGUF** per l'app Android.

**Impostazioni del notebook** (pannello a destra → *Settings*):
- *Accelerator*: **GPU T4 x2**
- *Internet*: **On** (richiede il numero di telefono verificato sull'account Kaggle)

Poi in alto a destra **Save Version → Save & Run All (Commit)**: il training gira da solo sui server
Kaggle (fino a 12 ore) anche se chiudi il browser. Quando ha finito, il file `vextor-q8_0.gguf`
è nella scheda **Output** della versione: scaricalo dal telefono e importalo in Vextor
(*Modelli → Scegli file .gguf*).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # una sola T4: più semplice e stabile
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Codice

In [ ]:
import os
REPO = "TiclyMusic/vextor1.0"
BRANCH = "ccr-4b3aab93-9puw42"  # metti "main" dopo aver unito il branch
WORK = "/tmp/vextor"
!rm -rf {WORK} && git clone -q --depth 1 -b {BRANCH} https://github.com/{REPO} {WORK}
%cd {WORK}
!pip install -q -r training/requirements.txt

## 2. Dataset (prodotto dalla GitHub Action, release `dataset-latest`)

In [ ]:
import urllib.request, tarfile
url = f"https://github.com/{REPO}/releases/download/dataset-latest/dataset.tar.gz"
urllib.request.urlretrieve(url, "dataset.tar.gz")
os.makedirs("data", exist_ok=True)
tarfile.open("dataset.tar.gz").extractall("data")
!cat data/dataset/stats.json
!wc -l data/dataset/*.jsonl

## 3. Training LoRA
Su una T4: circa 1 ora per epoca con ~1500 esempi.
Su Kaggle (12 ore) puoi fare 2 epoche. Se la sessione si interrompe, riesegui la cella: riparte dall'ultimo checkpoint (`--resume`).

In [ ]:
BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # oppure "Qwen/Qwen2.5-Coder-3B-Instruct" (più lento)
MAX_LEN = 6144
EPOCHS = 2
!python training/train_lora.py --base {BASE} --data data/dataset --out out/vextor-lora --max-len {MAX_LEN} --epochs {EPOCHS} --resume

## 4. Prova veloce

In [ ]:
import torch, re, sys
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from IPython.display import HTML, display
sys.path.insert(0, "scraper")
from build_dataset import SYSTEM_PROMPT

tok = AutoTokenizer.from_pretrained("out/vextor-lora")
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="cuda")
model = PeftModel.from_pretrained(model, "out/vextor-lora").eval()
msgs = [{"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Crea una landing page per un'app di meditazione, tema scuro, palette viola."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
out = model.generate(**ids, max_new_tokens=3000, temperature=0.3, top_p=0.9, do_sample=True)
answer = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
html = (re.search(r"```html\n(.*?)(```|$)", answer, re.S) or [None, answer])[1]
display(HTML(f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}" style="width:100%;height:600px;border:1px solid #ccc"></iframe>'))
del model; torch.cuda.empty_cache()

## 5. Export GGUF

In [ ]:
QUANT = "q8_0"   # "q4_k_m" = file più piccolo (~1 GB), compila llama.cpp
!python training/train_lora.py --merge --out out/vextor-lora --merged out/vextor-merged
!bash training/export_gguf.sh out/vextor-merged out/vextor-{QUANT}.gguf {QUANT}
GGUF = f"out/vextor-{QUANT}.gguf"

## 6. Output
Il file viene copiato in `/kaggle/working`, quindi compare nella scheda **Output** della versione salvata.

In [ ]:
!cp {GGUF} /kaggle/working/
!ls -lh /kaggle/working/*.gguf